In [1]:
import pandas as pd
import torch
import torch.nn as nn
import math
import numpy as np
import jieba
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader

In [2]:
df = pd.read_table('./data/cmn.txt', header=None)
df = df.drop(2, axis=1)

In [3]:
df.head()

,0,1
0,Hi.,嗨。
1,Hi.,你好。
2,Run.,你用跑的。
3,Stop!,住手！
4,Wait!,等等！


In [4]:
df[0] = df[0].str.lower()
df[0] = df[0].str.replace(r"([.!?])", r" \1", regex=True)
df[0] = df[0].str.replace(r"[^a-zA-Z.!?\s]", r" ", regex=True)
df[0] = df[0].str.replace(r"\s+", r" ", regex=True)
df[0] = df[0].str.strip()
df.head()

,0,1
0,hi .,嗨。
1,hi .,你好。
2,run .,你用跑的。
3,stop !,住手！
4,wait !,等等！


In [5]:

eng_sentences = df[0].tolist()
chn_sentences = df[1].tolist()


def build_vocab(sentences, is_chinese=False):
    word2id = {'<PAD>': 0, '<UNK>': 1, '<BOS>': 2, '<EOS>': 3}
    

    current_id = 4 

    for sentence in sentences:
        if is_chinese:
            tokens = jieba.lcut(sentence) 
        else:
            tokens = sentence.lower().split()
            
        for token in tokens:
            if token not in word2id:
                word2id[token] = current_id
                current_id += 1
        
    return word2id


eng2id = build_vocab(eng_sentences, is_chinese=False)
chn2id = build_vocab(chn_sentences, is_chinese=True)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\yixingll\AppData\Local\Temp\jieba.cache
Loading model cost 0.414 seconds.
Prefix dict has been built successfully.


In [6]:
class MyDataset(Dataset):
    def __init__(self, dataframe, eng2id, chn2id):
        super().__init__()
        self.dataframe = dataframe
        self.src_data = []
        for sentence in dataframe[0]:
            cur_sentence = []
            for word in sentence.split():
                if word in eng2id:
                    cur_sentence.append(eng2id[word])
                else:
                    cur_sentence.append(eng2id['<UNK>'])
            cur_sentence.append(3)
            cur_sentence = torch.tensor(cur_sentence, dtype=torch.long)
            self.src_data.append(cur_sentence)
            
        self.tgt_data = []
        for sentence in dataframe[1]:
            cur_sentence = [2]
            for word in jieba.lcut(sentence):
                if word in chn2id:
                    cur_sentence.append(chn2id[word])
                else:
                    cur_sentence.append(chn2id['<UNK>'])
            cur_sentence.append(3)
            cur_sentence = torch.tensor(cur_sentence, dtype=torch.long)
            self.tgt_data.append(cur_sentence)
    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.tgt_data[idx]
        

In [7]:
def collate_fn(batch):
    src_list = []
    tgt_list = []

    for src, tgt in batch:
        src_list.append(src)
        tgt_list.append(tgt)

    src_padded = pad_sequence(src_list, batch_first=True, padding_value=0)
    tgt_padded = pad_sequence(tgt_list, batch_first=True, padding_value=0)

    return src_padded, tgt_padded

In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_seq_len, d_model)
        
        position = torch.arange(0, max_seq_len).unsqueeze(1).float()

        div_term = torch.exp(torch.arange(0, d_model, 2).float() 
                             * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]

        return self.dropout(x)
        

In [9]:
def scaled_dot_product_attention(Q, K, V, mask=None):


    d_k = K.shape[-1]
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    attn_weights = torch.softmax(scores, dim=-1)
    output = torch.matmul(attn_weights, V)
    return output, attn_weights

In [10]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        Q = Q.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask=mask)

        concat_output = attn_output.transpose(1, 2).contiguous().view(
                        batch_size, -1, self.d_model)

        output = self.W_o(concat_output)

        return output, attn_weights

In [11]:
class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.fc2(self.dropout(self.relu(self.fc1(x))))
        return x

In [12]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output, _ = self.self_attn(x, x, x, mask=mask)
        x = self.norm1(x + self.dropout1(attn_output))
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output))
        return x

In [13]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, 
                 num_layers, dropout):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) 
            for _ in range(num_layers)
        ])

    def forward(self, X, mask):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.d_model))
        self.attn_weights = [None] * len(self.layers)
        for layer in self.layers:
            X = layer(X, mask)
        return X

In [14]:
def create_look_ahead_mask(size):
    mask = torch.ones(size, size)
    mask = torch.tril(mask)
    return mask

In [15]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout=dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        
        attn1, _ = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout1(attn1))
        
        attn2, _ = self.cross_attn(x, enc_output, enc_output, mask=src_mask)
        x = self.norm2(x + self.dropout2(attn2))
        

        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_output))
        
        return x

In [16]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, 
                 num_layers, dropout):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) 
            for _ in range(num_layers)
        ])

    def forward(self, X, enc_output, src_mask, tgt_mask):
        X = self.pos_encoding(self.embedding(X) * math.sqrt(self.d_model))
        self.attn_weights = [None] * len(self.layers) 
        for layer in self.layers:
            X = layer(X, enc_output, src_mask, tgt_mask) 
        return X

In [17]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512,
                 num_heads = 8, num_layers=6, d_ff=2048, dropout=0.1):
        super().__init__()

        self.encoder = Encoder(src_vocab_size, d_model, num_heads,
                               d_ff, num_layers, dropout)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_heads,
                               d_ff, num_layers, dropout)
        self.projection = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, src_mask, tgt_mask):
        enc_output = self.encoder(src, src_mask)

        dec_output = self.decoder(tgt, enc_output, src_mask, tgt_mask)

        logits = self.projection(dec_output)

        return logits

In [18]:
def make_src_mask(src):

    src_mask = (src != PAD_IDX).unsqueeze(1).unsqueeze(2)
    return src_mask

def make_tgt_mask(tgt):
    tgt_len = tgt.shape[1]

    tgt_pad_mask = tgt_mask = (tgt != PAD_IDX).unsqueeze(1).unsqueeze(2)
    tgt_sub_mask = torch.tril(torch.ones((tgt_len, tgt_len), device=tgt.device)).bool()
    
    tgt_mask = tgt_pad_mask & tgt_sub_mask
    
    return tgt_mask


In [19]:
class ScheduledOptim:
    def __init__(self, optimizer, d_model, warmup_steps):
        self._optimizer = optimizer
        self.d_model = d_model
        self.n_warmup_steps = warmup_steps
        self.n_current_steps = 0

    def zero_grad(self):
        self._optimizer.zero_grad()
        
    def step_and_update_lr(self):
        self._update_learning_rate()
        self._optimizer.step()

    def _get_lr_scale(self):
        
        scale = min(self.n_current_steps ** -0.5,
                    self.n_current_steps * self.n_warmup_steps ** -1.5)

        return (self.d_model ** -0.5) * scale
    def _update_learning_rate(self):

        self.n_current_steps += 1
        lr = self._get_lr_scale()

        for param_group in self._optimizer.param_groups:
            param_group['lr'] = lr

In [20]:
def train_step(model, src_batch, tgt_batch, optimizer, criterion):
    tgt_input = tgt_batch[:, :-1]
    tgt_label = tgt_batch[:, 1:]
    
    src_mask = make_src_mask(src_batch)
    tgt_mask = make_tgt_mask(tgt_input)

    optimizer.zero_grad()

    logits = model(src_batch, tgt_input, src_mask, tgt_mask)

    logits_flat = logits.contiguous().view(-1, logits.size(-1))
    tgt_label_flat = tgt_label.contiguous().view(-1)
    loss = criterion(logits_flat, tgt_label_flat)
    
    loss.backward()

    optimizer.step_and_update_lr()

    return loss.item()

In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device:{device}")

dataset = MyDataset(df, eng2id, chn2id)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

SRC_VOCAB_SIZE = len(eng2id)
TGT_VOCAB_SIZE = len(chn2id)
D_MODEL = 512
NUM_HEADS = 8
NUM_LAYERS = 6
D_FF = 2048
EPOCHS = 10

model = Transformer(
    src_vocab_size=SRC_VOCAB_SIZE, 
    tgt_vocab_size=TGT_VOCAB_SIZE, 
    d_model=D_MODEL, 
    num_heads=NUM_HEADS, 
    num_layers=NUM_LAYERS, 
    d_ff=D_FF
).to(device)

PAD_IDX = 0
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

base_optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
optimizer = ScheduledOptim(base_optimizer, D_MODEL, warmup_steps=4000)

for epoch in range(EPOCHS):

    model.train() 
    
    total_loss = 0
    

    for batch_idx, (src_batch, tgt_batch) in enumerate(dataloader):
        

        src_batch = src_batch.to(device)
        tgt_batch = tgt_batch.to(device)
        

        loss = train_step(model, src_batch, tgt_batch, optimizer, criterion)
        
        total_loss += loss
        

        if batch_idx % 50 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Batch {batch_idx} | 当前 Loss: {loss:.4f}")
            

    avg_loss = total_loss / len(dataloader)
    print(f"====== Epoch {epoch+1} 结束 | 平均 Loss: {avg_loss:.4f} ======")
    

    torch.save(model.state_dict(), f"transformer_epoch_{epoch+1}.pt")

device:cuda
Epoch 1/10 | Batch 0 | 当前 Loss: 9.7677
Epoch 1/10 | Batch 50 | 当前 Loss: 7.7811
Epoch 1/10 | Batch 100 | 当前 Loss: 6.8741
Epoch 1/10 | Batch 150 | 当前 Loss: 6.3488
Epoch 1/10 | Batch 200 | 当前 Loss: 5.9115
Epoch 1/10 | Batch 250 | 当前 Loss: 5.4069
Epoch 1/10 | Batch 300 | 当前 Loss: 5.4082
Epoch 1/10 | Batch 350 | 当前 Loss: 5.1807
Epoch 1/10 | Batch 400 | 当前 Loss: 4.8343
Epoch 1/10 | Batch 450 | 当前 Loss: 4.8562
Epoch 1/10 | Batch 500 | 当前 Loss: 4.9841
Epoch 1/10 | Batch 550 | 当前 Loss: 4.6738
Epoch 1/10 | Batch 600 | 当前 Loss: 4.3290
Epoch 1/10 | Batch 650 | 当前 Loss: 4.7296
Epoch 1/10 | Batch 700 | 当前 Loss: 4.0414
Epoch 1/10 | Batch 750 | 当前 Loss: 4.6685
Epoch 1/10 | Batch 800 | 当前 Loss: 4.6510
Epoch 1/10 | Batch 850 | 当前 Loss: 4.2094
Epoch 1/10 | Batch 900 | 当前 Loss: 4.5874
====== Epoch 1 结束 | 平均 Loss: 5.3048 ======
Epoch 2/10 | Batch 0 | 当前 Loss: 3.3522
Epoch 2/10 | Batch 50 | 当前 Loss: 3.9068
Epoch 2/10 | Batch 100 | 当前 Loss: 3.6428
Epoch 2/10 | Batch 150 | 当前 Loss: 3.5102
Epoch 2/

In [24]:
def translate(model, sentence, eng2id, id2chn, device, max_len=50):

    model.eval()
    

    sentence = sentence.lower().strip()

    import re
    sentence = re.sub(r"([.!?])", r" \1", sentence)

    tokens = sentence.split()
    src_ids = []
    for token in tokens:
        if token in eng2id:
            src_ids.append(eng2id[token])
        else:
            src_ids.append(eng2id['<UNK>'])
            
    src_ids.append(3)
    

    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(device)
    

    src_mask = make_src_mask(src_tensor)
    

    with torch.no_grad():
        enc_output = model.encoder(src_tensor, src_mask)
        

    tgt_ids = [2]
    

    for _ in range(max_len):

        tgt_tensor = torch.tensor([tgt_ids], dtype=torch.long).to(device)
        

        tgt_mask = make_tgt_mask(tgt_tensor)
        

        with torch.no_grad():
            dec_output = model.decoder(tgt_tensor, enc_output, src_mask, tgt_mask)

            logits = model.projection(dec_output[:, -1, :])
            

        next_word_id = torch.argmax(logits, dim=-1).item()
        

        tgt_ids.append(next_word_id)
        

        if next_word_id == 3:
            break
            

    translated_words = []
    for idx in tgt_ids:
        if idx not in [2, 3, 0]: 
            translated_words.append(id2chn.get(idx, '<UNK>'))
            

    return "".join(translated_words)


id2chn = {v: k for k, v in chn2id.items()}


test_sentences = [
    "Hi .",
    "Run .",
    "Stop !",
    "Wait !"
]


for eng_text in test_sentences:
    res = translate(model, eng_text, eng2id, id2chn, device)
    print(f"英: {eng_text}  --> 中: {res}")

英: Hi .  --> 中: 请你好。
英: Run .  --> 中: 跑步。
英: Stop !  --> 中: 别再埋怨了！
英: Wait !  --> 中: 等一下！


In [23]:
test_sentences = [
    "He is alone",
    "I Run",
    "Marry hate fish",
    "Tom feel happy"
]


for eng_text in test_sentences:
    res = translate(model, eng_text, eng2id, id2chn, device)
    print(f"英: {eng_text}  --> 中: {res}")

英: He is alone  --> 中: 他獨自一人。
英: I Run  --> 中: 我跑。
英: Marry hate fish  --> 中: 讨厌鱼。
英: Tom feel happy  --> 中: 湯姆感覺不高兴。
